In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18
import pyvo as vo
import requests
import sqlalchemy as sa
import sys

In [2]:
from IPython.core.display import HTML
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

In [3]:
from alerce.core import Alerce

In [4]:
alerce_client = Alerce()

In [5]:
per_other = alerce_client.query_objects(classifier="lc_classifier",
                           class_name="Periodic-Other",
                           order_by="probability",
                           order_mode="DESC",
                           page_size=3000, format='pandas')
print(per_other.shape)

(3000, 23)


In [6]:
per_other['meanra']

0       286.481815
1       287.252824
2       330.804710
3       107.754854
4       261.224804
           ...    
2995      6.888810
2996    108.114371
2997    200.291096
2998    262.532483
2999     86.628449
Name: meanra, Length: 3000, dtype: float64

In [7]:
per_other['meandec']

0       70.208211
1      -15.407508
2        2.822860
3       -4.048827
4        9.318029
          ...    
2995    24.759875
2996   -25.550290
2997    50.306689
2998    12.312180
2999    37.459317
Name: meandec, Length: 3000, dtype: float64

# GEMINI ----- IGNORAR

In [9]:
import pandas as pd
from alerce.core import Alerce

alerce_client = Alerce()

# 1. Extraemos las coordenadas de tus objetos ZTF
# Usamos una lista de diccionarios para iterar o consultas por lotes
lsst_matches = []

print("Iniciando búsqueda en LSST...")

# Nota: Para 3000 objetos, es mejor hacerlo por lotes o usar el servicio TAP
# Aquí un ejemplo de cómo consultarías los primeros para verificar:
for index, row in per_other.head(100).iterrows():
    # Buscamos en LSST usando un radio pequeño (ej. 1.5 arcsec)
    # En 2026, especificamos el catálogo de LSST si está disponible en el query
    res = alerce_client.query_objects(
        ra=row['meanra'], 
        dec=row['meandec'], 
        radius=1.5,
        format='pandas'
    )
    
    # Filtramos los resultados que pertenezcan a LSST (usando el sid que encontraste antes)
    if not res.empty:
        # Suponiendo que 'LSST' es el sid para Rubin
        match = res[res['sid'] == 'LSST'] 
        if not match.empty:
            match['ztf_oid'] = row['oid'] # Mantenemos la referencia al original
            lsst_matches.append(match)

# Consolidamos los resultados
df_final = pd.concat(lsst_matches) if lsst_matches else pd.DataFrame()
print(f"Encontrados {len(df_final)} matches en LSST.")

Iniciando búsqueda en LSST...


KeyError: 'sid'

In [10]:
# Definimos la consulta para buscar objetos LSST cerca de tus candidatos
# Suponiendo que 'per_other' es tu lista de ZTF
query = """
SELECT 
    lsst.oid AS lsst_id, 
    lsst.meanra, 
    lsst.meandec, 
    lsst.probability AS lsst_prob,
    ztf.oid AS ztf_id
FROM 
    lsst_alerce.object AS lsst
JOIN 
    ztf_alerce.object AS ztf 
    ON CONTAINS(POINT('ICRS', lsst.meanra, lsst.meandec), 
                CIRCLE('ICRS', ztf.meanra, ztf.meandec, 1.5/3600))
WHERE 
    ztf.oid IN ({})
""".format(", ".join(f"'{id}'" for id in per_other['oid'][:500])) # Procesamos de 500 en 500

df_matches = alerce_tap.search(query).to_table().to_pandas()
display(df_matches.head())

NameError: name 'alerce_tap' is not defined

# CROSS MATCHING

In [11]:
per_other.to_csv('3000.csv', index=False)

In [12]:
df = pd.read_csv('resultado_vs.csv')

In [13]:
df

,oid,probability,class_name,n_det,meanra,meandec
0,313664304786178164,0.499153,VS,46,60.315773,-50.512240
1,313664304777265264,0.551525,VS,79,59.006575,-49.892756
2,313637935749005346,0.490714,VS,29,58.323980,-49.267754
3,313664313447415929,0.557866,VS,66,61.217991,-47.808827
4,313664304784605245,0.554795,VS,13,60.410005,-50.449750
5,313664313833816089,0.594529,VS,59,59.021008,-49.550005
6,313664312965595139,0.569247,VS,70,61.587340,-47.435906
7,313695108419026949,0.543513,VS,47,63.153393,-48.981124
8,313695108169990145,0.584501,VS,10,64.634008,-46.214936
9,313708289179779073,0.513568,VS,26,62.824037,-46.258675


In [14]:
per_other

,oid,ndethist,ncovhist,mjdstarthist,mjdendhist,corrected,stellar,ndet,g_r_max,g_r_max_corr,g_r_mean,g_r_mean_corr,firstmjd,lastmjd,deltajd,meanra,meandec,sigmara,sigmadec,class,classifier,probability,step_id_corr
0,ZTF18abhzaws,370,3201,58263.450058,61072.519271,True,False,243,NaN,NaN,NaN,NaN,58568.430231,61072.519271,2504.089039,286.481815,70.208211,0.012086,0.004092,Periodic-Other,lc_classifier,0.870488,27.5.7a32.dev1
1,ZTF18abppvcc,731,3368,58292.313669,60978.098356,True,False,388,NaN,NaN,NaN,NaN,58593.439120,60978.098356,2384.659236,287.252824,-15.407508,0.003067,0.002957,Periodic-Other,lc_classifier,0.869120,27.5.7a32.dev1
2,ZTF18abmdvem,316,1675,58285.462500,60967.212616,True,True,151,NaN,NaN,NaN,NaN,58340.338194,60967.212616,2626.874421,330.804710,2.822860,0.005658,0.005651,Periodic-Other,lc_classifier,0.864864,27.5.6
3,ZTF18acpvnjk,286,1742,58439.454630,61106.187164,True,False,125,NaN,NaN,NaN,NaN,58441.461435,61106.187164,2664.725729,107.754854,-4.048827,0.006554,0.006538,Periodic-Other,lc_classifier,0.864576,27.5.7a32.dev1
4,ZTF18abrzhwr,370,2636,58258.356852,61107.404167,True,False,204,NaN,NaN,NaN,NaN,58362.208206,61107.404167,2745.195961,261.224804,9.318029,0.005263,0.005193,Periodic-Other,lc_classifier,0.862920,27.5.7a32.dev1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,ZTF18adbmjcj,116,3465,58482.154225,60667.108009,True,True,35,NaN,NaN,NaN,NaN,58482.154225,60667.108009,2184.953785,6.888810,24.759875,0.013542,0.012297,Periodic-Other,lc_classifier,0.760608,26.0.1
2996,ZTF18adauhtj,967,2783,58461.432755,61108.147419,True,False,324,NaN,NaN,NaN,NaN,58539.211308,61108.147419,2568.936111,108.114371,-25.550290,0.004556,0.004110,Periodic-Other,lc_classifier,0.760608,27.5.7a32.dev1
2997,ZTF18aagxzal,1529,6079,58174.508090,61076.495683,True,False,621,NaN,NaN,NaN,NaN,58300.198171,61076.495683,2776.297512,200.291096,50.306689,0.004398,0.002809,Periodic-Other,lc_classifier,0.760608,27.5.7a32.dev1
2998,ZTF18admncfx,540,2536,58344.159607,60902.246076,True,False,68,NaN,NaN,NaN,NaN,59673.436111,60902.246076,1228.809965,262.532483,12.312180,0.009063,0.008855,Periodic-Other,lc_classifier,0.760608,27.5.6


In [17]:
from astropy.coordinates import SkyCoord
from astropy import units as u
df_ztf = per_other
df_lsst = df


# Definir las coordenadas de cada tabla
# Nota: asegúrate de que los nombres de las columnas coincidan (ra, dec)
coord_ztf = SkyCoord(ra=df_ztf['meanra'].values * u.deg, dec=df_ztf['meandec'].values * u.deg)
coord_lsst = SkyCoord(ra=df_lsst['meanra'].values * u.deg, dec=df_lsst['meandec'].values * u.deg)

# Buscar los vecinos más cercanos en LSST para cada objeto en ZTF
# Usamos un radio de 1.5 arcsec, que es el estándar para catálogos modernos
idx, d2d, d3d = coord_ztf.match_to_catalog_sky(coord_lsst)

# Filtrar resultados por un radio de separación máximo
max_sep = 10 * u.arcsec
sep_constraint = d2d < max_sep

# Crear un DataFrame solo con los matches válidos
df_matches = pd.DataFrame({
    'ztf_oid': df_ztf.loc[sep_constraint, 'oid'].values,
    'lsst_oid': df_lsst.iloc[idx[sep_constraint]]['oid'].values,
    'separation_arcsec': d2d[sep_constraint].to(u.arcsec).value
})

print(f"Se encontraron {len(df_matches)} objetos en común.")

Se encontraron 0 objetos en común.
